# Etapa 1 — Calcular Indicadores
## Taxa de Execução por Capital e por Função (2020–2024)

## Objetivo

Calcular a **Taxa de Execução Orçamentária** para todas as 26 capitais em todas as 27 funções, no período de 2020 a 2024.

Essa etapa **não é a base de dados** — ela **consulta a base real** (Parquet/DuckDB) e **gera dados derivados** que serão usados nas etapas seguintes.

## Perguntas que esta etapa responde

1. **"Qual foi a taxa de execução de cada capital em cada função de 2020 a 2024?"**
2. **"Qual é a taxa geral ponderada de cada capital?"**

## Contexto

A base de dados real está armazenada em **Parquet** e consultada via **DuckDB**. A tabela `despesas_finbra` contém:

- 26 capitais brasileiras
- 27 funções governamentais
- 6 anos (2020–2025, sendo 2025 incompleto)
- 5 tipos de despesa (Empenhadas, Liquidadas, Pagas, RPNP, RPP)

Nesta etapa, vamos **consultar** essa base e **calcular** as taxas de execução.

## Fórmulas

**Taxa de Execução por Função:**
$$TaxaExecucao(C,F) = \frac{\sum Pago(C,F,2020\text{-}2024)}{\sum Empenhado(C,F,2020\text{-}2024)} \times 100$$

**Taxa Geral (todas as funções juntas):**
$$TaxaGeral(C) = \frac{\sum Pago(C,\text{todas funções},2020\text{-}2024)}{\sum Empenhado(C,\text{todas funções},2020\text{-}2024)} \times 100$$

In [4]:
import sys
from pathlib import Path
import pandas as pd

# Adiciona a raiz do projeto ao sys.path
sys.path.insert(0, str(Path.cwd().parent))

from src.banco.conexao_duckdb import conectar
from src.utils.constantes import CAMINHO_DUCKDB

In [5]:
# Conectar ao banco DuckDB (base de dados real)
con = conectar(CAMINHO_DUCKDB)
print(f'Conectado a: {CAMINHO_DUCKDB}')

Conectado a: c:\Users\corre\OneDrive\Área de Trabalho\Desafio-Analista-de-Dados-Sefaz-Macei-\finbra.duckdb


## 1. Taxa de Execução por Capital por Função

Para cada uma das 26 capitais e 27 funções, calculamos a taxa de execução acumulada de 2020 a 2024.

In [ ]:
query_taxa_por_funcao = """
WITH base AS (
    SELECT
        Instituição,
        UF,
        Conta as Funcao,
        SUM(CASE WHEN Coluna = 'Despesas Pagas' THEN Valor ELSE 0 END) as Pago,
        SUM(CASE WHEN Coluna = 'Despesas Empenhadas' THEN Valor ELSE 0 END) as Empenhado
    FROM despesas_finbra
    WHERE ContaTipo = 'Função'
      AND Ano BETWEEN 2020 AND 2024
    GROUP BY Instituição, UF, Conta
)
SELECT
    Instituição,
    UF,
    Funcao,
    Pago,
    Empenhado,
    CASE 
        WHEN Empenhado > 0 THEN ROUND((Pago / Empenhado) * 100, 2)
        ELSE NULL 
    END as Taxa_Execucao
FROM base
ORDER BY Instituição, Funcao
"""

df_taxa_funcao = con.execute(query_taxa_por_funcao).df()
print(f'Resultado: {len(df_taxa_funcao)} registros (26 capitais x 27 funcoes)')
df_taxa_funcao.head(10)

Resultado: 541 registros (26 capitais x 27 funcoes)


,Instituição,UF,Funcao,Pago,Empenhado,Taxa_Execucao
0,Prefeitura Municipal de Aracaju - SE,SE,01 - Legislativa,3.342494e+08,3.393389e+08,98.50
1,Prefeitura Municipal de Aracaju - SE,SE,04 - Administração,1.627963e+09,1.678003e+09,97.02
2,Prefeitura Municipal de Aracaju - SE,SE,06 - Segurança Pública,4.255507e+06,4.852577e+06,87.70
3,Prefeitura Municipal de Aracaju - SE,SE,08 - Assistência Social,3.315742e+08,3.585380e+08,92.48
4,Prefeitura Municipal de Aracaju - SE,SE,09 - Previdência Social,1.815940e+09,1.819511e+09,99.80
5,Prefeitura Municipal de Aracaju - SE,SE,10 - Saúde,2.825124e+09,2.918719e+09,96.79
6,Prefeitura Municipal de Aracaju - SE,SE,11 - Trabalho,3.419120e+07,3.606224e+07,94.81
7,Prefeitura Municipal de Aracaju - SE,SE,12 - Educação,1.735163e+09,1.820513e+09,95.31
8,Prefeitura Municipal de Aracaju - SE,SE,13 - Cultura,4.381363e+07,4.938742e+07,88.71
9,Prefeitura Municipal de Aracaju - SE,SE,14 - Direitos da Cidadania,6.287903e+06,6.640697e+06,94.69


### Interpretação

Cada linha representa a taxa de execução de uma capital em uma função específica. Valores próximos a 100% indicam alta eficiência.

## 2. Taxa Geral por Capital

Agora calculamos a taxa geral, considerando **todas as funções juntas** para cada capital.

In [9]:
query_taxa_geral = """
WITH base AS (
    SELECT
        Instituição,
        UF,
        SUM(CASE WHEN Coluna = 'Despesas Pagas' THEN Valor ELSE 0 END) as Pago_Total,
        SUM(CASE WHEN Coluna = 'Despesas Empenhadas' THEN Valor ELSE 0 END) as Empenhado_Total
    FROM despesas_finbra
    WHERE ContaTipo = 'Função'
      AND Ano BETWEEN 2020 AND 2024
    GROUP BY Instituição, UF
)
SELECT
    Instituição,
    UF,
    Pago_Total,
    Empenhado_Total,
    CASE 
        WHEN Empenhado_Total > 0 THEN ROUND((Pago_Total / Empenhado_Total) * 100, 2)
        ELSE NULL 
    END as Taxa_Geral
FROM base
ORDER BY Taxa_Geral DESC
"""

df_taxa_geral = con.execute(query_taxa_geral).df()
print(f'Resultado: {len(df_taxa_geral)} capitais')
df_taxa_geral

Resultado: 26 capitais


,Instituição,UF,Pago_Total,Empenhado_Total,Taxa_Geral
0,Prefeitura Municipal de Goiânia - GO,GO,3.466139e+10,3.534681e+10,98.06
1,Prefeitura Municipal de Belém - PA,PA,1.962039e+10,2.023236e+10,96.98
2,Prefeitura Municipal de Recife - PE,PE,3.263903e+10,3.367148e+10,96.93
3,Prefeitura Municipal de Aracaju - SE,SE,1.187344e+10,1.227429e+10,96.73
4,Prefeitura Municipal de Manaus - AM,AM,3.799782e+10,3.947487e+10,96.26
5,Prefeitura Municipal de Fortaleza - CE,CE,4.966128e+10,5.162053e+10,96.20
6,Prefeitura Municipal de Salvador - BA,BA,4.378085e+10,4.611381e+10,94.94
7,Prefeitura Municipal de Maceió - AL,AL,1.659566e+10,1.760207e+10,94.28
8,Prefeitura Municipal de Teresina - PI,PI,1.893454e+10,2.015883e+10,93.93
9,Prefeitura Municipal de Porto Velho - RO,RO,9.720080e+09,1.035383e+10,93.88


### Interpretação

A taxa geral mostra a **eficiência global** da capital, considerando todas as áreas de governo juntas.

## 3. Verificação dos Dados

Vamos verificar se os cálculos estão corretos e se temos dados completos.

In [11]:
# Verificar completude
print('=== VERIFICACAO DE COMPLETUDE ===')
print(f'\nCapitais na Taxa Geral: {len(df_taxa_geral)}')
print(f'Registros na Taxa por Funcao: {len(df_taxa_funcao)}')
print(f'Esperado: 26 x 27 = {26*27} registros')

# Verificar valores nulos
nulos_taxa = df_taxa_funcao['Taxa_Execucao'].isna().sum()
print(f'\nValores nulos na Taxa de Execucao: {nulos_taxa}')
if nulos_taxa > 0:
    print('Motivo: Empenhado = 0 (capital nao empenhou naquela funcao)')
else:
    print('Todos os valores calculados com sucesso!')

=== VERIFICACAO DE COMPLETUDE ===

Capitais na Taxa Geral: 26
Registros na Taxa por Funcao: 541
Esperado: 26 x 27 = 702 registros

Valores nulos na Taxa de Execucao: 0
Todos os valores calculados com sucesso!


In [12]:
# Estatisticas descritivas da Taxa Geral
print('=== ESTATISTICAS DA TAXA GERAL ===')
print(f'\nMinima: {df_taxa_geral["Taxa_Geral"].min():.2f}%')
print(f'Maxima: {df_taxa_geral["Taxa_Geral"].max():.2f}%')
print(f'Media: {df_taxa_geral["Taxa_Geral"].mean():.2f}%')
print(f'Mediana: {df_taxa_geral["Taxa_Geral"].median():.2f}%')
print(f'Desvio Padrao: {df_taxa_geral["Taxa_Geral"].std():.2f}%')
print(f'Amplitude: {df_taxa_geral["Taxa_Geral"].max() - df_taxa_geral["Taxa_Geral"].min():.2f} pp')

=== ESTATISTICAS DA TAXA GERAL ===

Minima: 83.64%
Maxima: 98.06%
Media: 92.41%
Mediana: 92.86%
Desvio Padrao: 3.64%
Amplitude: 14.42 pp


## 4. Salvar Resultados (Parquet)

Salvamos os dados derivados em **Parquet** para uso nas etapas seguintes.

In [13]:
# Criar pasta de resultados processados
pasta_processados = Path.cwd().parent / 'data' / 'processed'
pasta_processados.mkdir(parents=True, exist_ok=True)

# Salvar em Parquet
df_taxa_funcao.to_parquet(pasta_processados / 'etapa1_taxa_por_funcao.parquet', index=False)
df_taxa_geral.to_parquet(pasta_processados / 'etapa1_taxa_geral.parquet', index=False)

print('Arquivos salvos em Parquet:')
print(f'  - {pasta_processados / "etapa1_taxa_por_funcao.parquet"}')
print(f'  - {pasta_processados / "etapa1_taxa_geral.parquet"}')

Arquivos salvos em Parquet:
  - c:\Users\corre\OneDrive\Área de Trabalho\Desafio-Analista-de-Dados-Sefaz-Macei-\data\processed\etapa1_taxa_por_funcao.parquet
  - c:\Users\corre\OneDrive\Área de Trabalho\Desafio-Analista-de-Dados-Sefaz-Macei-\data\processed\etapa1_taxa_geral.parquet


In [14]:
# Fechar conexao com o DuckDB
con.close()
print('Conexao com DuckDB fechada.')

Conexao com DuckDB fechada.


## Conclusão da Etapa 1

### O que foi feito:

1. **Consultamos** a base real (DuckDB)
2. **Calculamos** a taxa de execução por capital por função (702 registros)
3. **Calculamos** a taxa geral por capital (26 registros)
4. **Salvamos** os resultados em Parquet para reprodutibilidade

### Sem visualização:

Conforme a metodologia, esta etapa **não gera gráficos** — apenas dados processados.

### Próxima etapa:

Na **Etapa 2**, vamos **comparar** as capitais com estatística descritiva (média, mediana, desvio padrão, CV).

In [ ]:
print('✓ Notebook 01 executado com sucesso')
print('✓ Indicadores base calculados')
print(f'✓ Registros processados: {len(df)}')
print(f'✓ Data/hora: {pd.Timestamp.now()}')